# ViT Analysis Figures

This notebook is intended to reproduce the ViT plots in the paper. It assumes that ViT representations and layerwise metrics have already been generated by the corresponding scripts (see details below).

Prerequisite metric are read from `results/vits/metrics/<method>/`, and figures are saved to `results/vits/figs/`.

## Prerequisite: ViT Reps Extraction and Metrics Computation

Example commands for one model:

```bash
uv run python scripts/vit_analysis/extract_vit_representations.py   --model-name google/vit-base-patch16-224   --model-key vit-base   --dataset imagenet7   --category-tag mix   --nsamples 5000   --num-workers 4

uv run python scripts/vit_analysis/compute_layerwise_metrics_vits.py --model vit-base --dataset imagenet7 --category mix --method gride
uv run python scripts/vit_analysis/compute_layerwise_metrics_vits.py --model vit-base --dataset imagenet7 --category mix --method entropy
uv run python scripts/vit_analysis/compute_layerwise_metrics_vits.py --model vit-base --dataset imagenet7 --category mix --method avg_l2
uv run python scripts/vit_analysis/compute_layerwise_metrics_vits.py --model vit-base --dataset imagenet7 --category mix --method avg_cosine
```

Run the analogous metric commands for `dinov3-vitb16` and `dinov3-vitl16` before executing all plotting cells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from rethinking_neural_id.paths import RepoPaths
from rethinking_neural_id.plotting.vit import (
    DEFAULT_VIT_MODELS,
    MODEL_COLORS,
    layer_slice,
    relative_depth,
    vit_figs_dir,
    load_avg_cosine,
    load_avg_l2,
    load_entropy,
    load_gride,
)

paths = RepoPaths.default()
fig_dir = vit_figs_dir(paths)

models = list(DEFAULT_VIT_MODELS)
dataset = "imagenet7"
category = "mix"
artifact_prefix = f"{dataset}-{category}"

print(f"Loading ViT metrics from: {paths.vit_metrics_root}")
print(f"Saving figures to: {fig_dir}")


def save_figure(fig, filename: str):
    output_path = fig_dir / filename
    fig.savefig(output_path, bbox_inches="tight")
    print(f"Saved: {output_path}")
    return output_path

## Average Representation Length

In [ ]:
sl = layer_slice(exclude_first=False)
line_w = 2
alpha_ci = 0.15
ms = 0
fs = 15

fig, ax = plt.subplots(figsize=(7, 4))

for model in models:
    mean, std = load_avg_l2(model, dataset=dataset, category=category, paths=paths, sl=sl)
    rel = relative_depth(len(mean))
    color = MODEL_COLORS[model]

    ax.plot(
        rel,
        mean,
        linewidth=line_w,
        alpha=0.98,
        color=color,
        marker="o" if ms > 0 else None,
        markersize=ms,
        label=model,
    )
    ax.fill_between(
        rel,
        mean - 2.0 * std,
        mean + 2.0 * std,
        color=color,
        alpha=alpha_ci,
        linewidth=0,
    )

ax.set_xlabel("Relative Depth", fontsize=fs)
ax.set_ylabel("Avg. representation length", fontsize=fs)
ax.tick_params(labelsize=fs - 1)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", alpha=0.25)

legend_handles = [
    Line2D(
        [0], [0], marker="o" if ms > 0 else None, linestyle="-",
        linewidth=line_w, markersize=ms if ms > 0 else 6,
        color=MODEL_COLORS[model], label=model,
    )
    for model in models
]
ax.legend(
    handles=legend_handles,
    labels=models,
    fontsize=fs - 1,
    title="Model",
    title_fontsize=fs - 1,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True,
)
fig.tight_layout()

save_figure(fig, f"{artifact_prefix}_avg_l2_vits.pdf")
plt.show()

## NN-Distances and Average Representation Length

In [ ]:
sl_gride = layer_slice(exclude_first=True)
sl_l2 = layer_slice(exclude_first=True)
line_w_avg = 2
line_w_scaling = 2
alpha_scaling = 0.4
alpha_ci = 0.15
ms = 0
fs = 15

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(14, 4))

for model in models:
    gride = load_gride(model, dataset=dataset, category=category, paths=paths, sl=sl_gride)
    r_ls = gride["r_ls"]
    r_avg = gride["r"]
    if np.all(np.isnan(r_avg)):
        r_avg = np.nanmean(r_ls, axis=1)

    L, S = r_ls.shape
    rel = relative_depth(L)
    color = MODEL_COLORS.get(model, ax_left._get_lines.get_next_color())

    for s in range(S):
        ax_left.plot(
            rel,
            r_ls[:, s],
            linewidth=line_w_scaling,
            alpha=alpha_scaling,
            color=color,
            linestyle=":" if s == 0 else "-",
            zorder=1,
        )

    ax_left.plot(
        rel,
        r_avg,
        linewidth=line_w_avg,
        alpha=0.98,
        color=color,
        label=model,
        marker="o" if ms > 0 else None,
        markersize=ms,
        zorder=3,
    )

for model in models:
    mean, std = load_avg_l2(model, dataset=dataset, category=category, paths=paths, sl=sl_l2)
    rel = relative_depth(len(mean))
    color = MODEL_COLORS.get(model, ax_right._get_lines.get_next_color())

    ax_right.plot(
        rel,
        mean,
        linewidth=line_w_avg,
        alpha=0.98,
        color=color,
        label=model,
        marker="o" if ms > 0 else None,
        markersize=ms,
        zorder=3,
    )
    ax_right.fill_between(
        rel,
        mean - 2.0 * std,
        mean + 2.0 * std,
        color=color,
        alpha=alpha_ci,
        linewidth=0,
        zorder=2,
    )

ax_left.set_xlabel("Relative Depth", fontsize=fs)
ax_left.set_ylabel("NN-Distances", fontsize=fs)
ax_left.tick_params(labelsize=fs - 1)
ax_left.grid(axis="y", alpha=0.25)
ax_left.grid(axis="x", alpha=0.25)

ax_right.set_xlabel("Relative Depth", fontsize=fs)
ax_right.set_ylabel("Avg. representation length", fontsize=fs)
ax_right.tick_params(labelsize=fs - 1)
ax_right.grid(axis="y", alpha=0.25)
ax_right.grid(axis="x", alpha=0.25)

legend_handles = [
    Line2D(
        [0], [0], marker="o" if ms > 0 else None, linestyle="-",
        linewidth=line_w_avg, markersize=ms if ms > 0 else 6,
        color=MODEL_COLORS[model], label=model,
    )
    for model in models
]
plt.subplots_adjust(right=0.82, wspace=0.28)
fig.legend(
    handles=legend_handles,
    labels=models,
    fontsize=fs - 1,
    title_fontsize=fs - 1,
    loc="center left",
    bbox_to_anchor=(0.86, 0.5),
    frameon=False,
)
fig.tight_layout(rect=[0, 0, 0.865, 1], w_pad=2.5)

save_figure(fig, f"{artifact_prefix}_nn_and_l2_vits.pdf")
plt.show()

## Average Cosine Similarity

In [ ]:
sl = layer_slice(exclude_first=True)
fs = 18
lw = 2
alpha_line = 0.9
alpha_ci = 0.15
ms = 8

fig, ax = plt.subplots(figsize=(13, 6.0))

for model in models:
    mean, std = load_avg_cosine(model, dataset=dataset, category=category, paths=paths, sl=sl)
    rel = relative_depth(len(mean))
    color = MODEL_COLORS[model]

    ax.plot(
        rel,
        mean,
        "-o",
        linewidth=lw,
        markersize=ms,
        alpha=alpha_line,
        color=color,
        label=model,
    )
    ax.fill_between(
        rel,
        mean - 2.0 * std,
        mean + 2.0 * std,
        color=color,
        alpha=alpha_ci,
        linewidth=0,
    )

ax.grid(axis="both", alpha=0.3)
ax.tick_params(labelsize=fs - 2)
ax.set_xlabel("Relative Depth", fontsize=fs)
ax.set_ylabel("Average Cosine Similarity", fontsize=fs)
ax.legend(
    fontsize=fs - 1,
    title_fontsize=fs - 1,
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False,
)
fig.tight_layout()

save_figure(fig, f"{artifact_prefix}_avg_cosine_vits.pdf")
plt.show()

## ID and Entropy Across Layers

In [ ]:
sl = layer_slice(exclude_first=True)
fs = 18
line_w_id = 2.8
line_w_ent = 2.2
alpha_line = 0.7

fig, ax_id = plt.subplots(figsize=(12.5, 6.0))
ax_ent = ax_id.twinx()
handles_models = []

for model in models:
    color = MODEL_COLORS.get(model, ax_id._get_lines.get_next_color())
    id_avg = load_gride(model, dataset=dataset, category=category, paths=paths, sl=sl)["id"]
    entropy = load_entropy(model, dataset=dataset, category=category, paths=paths, sl=sl)

    L = int(min(len(id_avg), len(entropy)))
    if L == 0:
        continue
    id_avg = id_avg[:L]
    entropy = entropy[:L]
    rel = relative_depth(L)

    id_line, = ax_id.plot(
        rel,
        id_avg,
        color=color,
        linewidth=line_w_id,
        alpha=alpha_line,
        label=model,
    )
    ax_ent.plot(
        rel,
        entropy,
        color=color,
        linewidth=line_w_ent,
        alpha=alpha_line,
        linestyle="--",
    )
    handles_models.append(id_line)

ax_id.set_xlabel("Relative Depth", fontsize=fs)
ax_id.set_ylabel("ID estimate (solid)", fontsize=fs)
ax_ent.set_ylabel("Entropy (dashed)", fontsize=fs)
ax_id.tick_params(labelsize=fs - 1)
ax_ent.tick_params(labelsize=fs - 1)
ax_id.grid(axis="y", alpha=0.25)
ax_id.grid(axis="x", alpha=0.25)

fig.subplots_adjust(right=0.76)
ax_id.legend(
    handles_models,
    [handle.get_label() for handle in handles_models],
    title="",
    fontsize=fs - 1,
    title_fontsize=fs - 1,
    loc="center left",
    bbox_to_anchor=(1.12, 0.55),
    frameon=False,
)
fig.tight_layout()

save_figure(fig, f"{artifact_prefix}_id_vs_entropy_vits.pdf")
plt.show()